In [2]:
import pandas as pd
import geopandas as gpd
import fiona
import numpy as np
import requests

This script starts from the list of fire we want to vaidate and sends of all the requests to the API

In [11]:
calfire_filtered_parks = gpd.read_file("Validation_Fire_Perimeters_2015_2024.shp")
calfire_filtered_parks.head()

,YEAR_,STATE,AGENCY,UNIT_ID,FIRE_NAME,INC_NUM,ALARM_DATE,CONT_DATE,CAUSE,C_METHOD,OBJECTIVE,GIS_ACRES,COMMENTS,COMPLEX_NA,IRWINID,FIRE_NUM,COMPLEX_ID,DECADES,geometry
0,2024,CA,NPS,KNP,COFFEE POT,00000088,2024/08/03 00:00:00,2024/12/16 00:00:00,1,3,1,14103.9000,None,None,{62A5DB78-38A8-4B54-8F7C-664DB0783E9C},None,None,2020-January 2025,"MULTIPOLYGON (((-13217941.346 4353257.821, -13..."
1,2024,CA,NPS,KNP,SENTINEL,00000058,2024/07/14 00:00:00,2024/11/08 00:00:00,14,7,1,260.6600,None,None,{5656FBF0-3D2E-4221-BA42-39E7723EC166},None,None,2020-January 2025,"POLYGON ((-13208263.579 4405096.209, -13208266..."
2,2024,CA,NPS,KNP,SIMPSON,00000074,2024/07/26 00:00:00,2024/09/02 00:00:00,14,2,1,52.1125,None,None,{65AA08F3-7325-4D42-8457-A6F0E12230AB},None,None,2020-January 2025,"POLYGON ((-13204355.841 4436805.636, -13204311..."
3,2023,CA,NPS,MNP,YORK,00010701,2023/07/28 00:00:00,2023/08/20 00:00:00,14,7,1,93077.9000,None,None,{B9E0F397-DE63-4B90-B5DA-9D04A7D2381B},None,None,2020-January 2025,"POLYGON ((-12815785.524 4224752.868, -12815785..."
4,2023,CA,NPS,KNP,REDWOOD,00000061,2023/08/15 00:00:00,2023/12/14 00:00:00,1,7,2,2248.4500,None,None,{405281C7-2B67-43C1-8F7A-F907AE53D92E},None,None,2020-January 2025,"POLYGON ((-13205607.284 4375004.509, -13205597..."


### Define functions

In [16]:

def create_bbox(fire_name, fires):
    ''' creates a buffered bounding box around a fire parameter '''

    fire = fires[fires['FIRE_NAME'] == fire_name].buffer(250).to_crs(epsg=4326)
    
    bbox = fire.bounds

    return bbox

def get_fire_date(fire_name, fires):
    ''' gets the alarm (start) date of a fire '''

    start_date = pd.to_datetime(fires[fires['FIRE_NAME'] == fire_name]['ALARM_DATE'].values[0]).to_datetime64()

    return start_date

def get_cont_date(fire_name, fires):
    ''' gets the containment date of a fire, returns None if missing '''

    raw = fires[fires['FIRE_NAME'] == fire_name]['CONT_DATE'].values[0]

    if pd.isnull(raw):
        return None

    return pd.to_datetime(raw).to_datetime64()

def create_query(fire_name, bbox, date_of_fire, post_fire_range, date_mode='alarm'):
    ''' Creates an API query for the bounding box and time period specified.
    
    date_mode: 'alarm' or 'cont' — recorded in the fire_event_name for downstream tracking.
    '''

    pre_start = date_of_fire - np.timedelta64(21, 'D')
    pre_end = date_of_fire - np.timedelta64(1, 'D')
    post_end = date_of_fire + np.timedelta64(post_fire_range, 'D')

    api_request = {
        "fire_event_name": f"{fire_name}_date{str(date_of_fire).split('T')[0]}_range{post_fire_range}_mode{date_mode}",
        "coarse_geojson": {
            "type": "Polygon",
            "coordinates": [[
                [float(bbox[0]), float(bbox[1])], 
                [float(bbox[2]), float(bbox[1])],
                [float(bbox[2]), float(bbox[3])], 
                [float(bbox[0]), float(bbox[3])], 
                [float(bbox[0]), float(bbox[1])]  
            ]]
        },
        "prefire_date_range": [str(pre_start).split('T')[0], str(pre_end).split('T')[0]],
        "postfire_date_range": [str(date_of_fire).split('T')[0], str(post_end).split('T')[0]]
    }
   
    return api_request


### Test if it works

In [17]:
url = "https://fire-recovery-backend-dev-113009620257.us-central1.run.app/fire-recovery/process/analyze_fire_severity" 

fire_name = 'YORK'
bbox = create_bbox(fire_name, calfire_filtered_parks)
date_of_fire = get_fire_date(fire_name, calfire_filtered_parks)
query_data = create_query(fire_name, bbox.values[0], date_of_fire, 15)
try:
    # Use the 'json' parameter: it automatically sets 'Content-Type: application/json'
    # and runs json.dumps() for you.
    response = requests.post(url, json=query_data)

    # 4. Check the results
    if response.status_code == 200 or response.status_code == 201:
        print("Success!")
        print(response.json()) # This is the data the API sends back
    else:
        print(f"Failed with status code: {response.status_code}")
        print(response.text) # This shows the error message from the API

except requests.exceptions.RequestException as e:
    print(f"A connection error occurred: {e}")

/opt/miniconda3/envs/burnseverity/lib/python3.13/site-packages/shapely/constructive.py:246: RuntimeWarning: invalid value encountered in buffer
  return lib.buffer(


Success!
{'fire_event_name': 'YORK_date2023-07-28_range15_modealarm', 'status': 'Processing started', 'job_id': '8536bc78-ed44-4bc6-a1bd-22aa0b02dff7'}


In [ ]:
# Get list of fire names to process
fire_names = calfire_filtered_parks['FIRE_NAME'].unique()

# Define post-fire day ranges to test
post_fire_days = [5, 10, 15, 21, 30, 45, 60, 90]

# date_modes: 'alarm' uses ALARM_DATE as the reference, 'cont' uses CONT_DATE
date_modes = ['alarm', 'cont']

# Initialize list to collect results
results = []

url = "https://fire-recovery-backend-dev-113009620257.us-central1.run.app/fire-recovery/process/analyze_fire_severity"

# Loop through each fire, date mode, and post-fire day range
for fire_name in fire_names:
    bbox = create_bbox(fire_name, calfire_filtered_parks)

    for date_mode in date_modes:
        if date_mode == 'alarm':
            date_of_fire = get_fire_date(fire_name, calfire_filtered_parks)
        else:
            date_of_fire = get_cont_date(fire_name, calfire_filtered_parks)

        if date_of_fire is None:
            for post_fire_range in post_fire_days:
                results.append({
                    'fire_event_name': None,
                    'job_id': None,
                    'fire_name': fire_name,
                    'date_mode': date_mode,
                    'post_fire_days': post_fire_range,
                    'status': 'skipped_no_cont_date'
                })
            print(f"- {fire_name} ({date_mode}): skipped — no CONT_DATE")
            continue

        for post_fire_range in post_fire_days:
            query_data = create_query(fire_name, bbox.values[0], date_of_fire, post_fire_range, date_mode)
            
            try:
                response = requests.post(url, json=query_data)
                
                if response.status_code == 200 or response.status_code == 201:
                    response_data = response.json()
                    results.append({
                        'fire_event_name': response_data.get('fire_event_name'),
                        'job_id': response_data.get('job_id'),
                        'fire_name': fire_name,
                        'date_mode': date_mode,
                        'post_fire_days': post_fire_range,
                        'status': 'success'
                    })
                    print(f"✓ {fire_name} ({date_mode}, {post_fire_range} days): {response_data.get('job_id')}")
                else:
                    results.append({
                        'fire_event_name': None,
                        'job_id': None,
                        'fire_name': fire_name,
                        'date_mode': date_mode,
                        'post_fire_days': post_fire_range,
                        'status': f'failed_{response.status_code}'
                    })
                    print(f"✗ {fire_name} ({date_mode}, {post_fire_range} days): Failed with {response.status_code}")
                    
            except requests.exceptions.RequestException as e:
                results.append({
                    'fire_event_name': None,
                    'job_id': None,
                    'fire_name': fire_name,
                    'date_mode': date_mode,
                    'post_fire_days': post_fire_range,
                    'status': 'error'
                })
                print(f"✗ {fire_name} ({date_mode}, {post_fire_range} days): Connection error")

# Create dataframe and save to CSV
df_results = pd.DataFrame(results)
df_results.to_csv('fire_processing_jobs.csv', index=False)

print(f"\nProcessed {len(results)} requests. Results saved to fire_processing_jobs.csv")
display(df_results)


/opt/miniconda3/envs/burnseverity/lib/python3.13/site-packages/shapely/constructive.py:246: RuntimeWarning: invalid value encountered in buffer
  return lib.buffer(


✓ COFFEE POT (alarm, 5 days): 674b9f1e-0dd5-43b6-9717-998a5c8b10e9
✓ COFFEE POT (alarm, 10 days): 36dce962-a982-4e60-af53-eb061e0c6428
